# Trees on the AI HAT

Turns the survey car's own detections into a model the **Raspberry Pi AI HAT+ (Hailo-8)** can run.

The Pi cannot do this itself: Hailo's compiler runs only on **x86_64 Linux**, which is what Colab is.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

About 10 min setup, 30–60 min training, 20–40 min compiling.

## 1. Check the machine

Must say `x86_64`, and should show a GPU.

In [ ]:
!uname -m && echo "--- must say x86_64 ---"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU: Runtime > Change runtime type > T4 GPU"
!free -g | awk 'NR==2{print "RAM: "$2" GB (compiler wants >= 12)"}'

## 2. Get the dataset in — via Google Drive

**Put `grove_dataset.tgz` in your Google Drive first**, then run the cell below.

Why Drive and not a direct Colab upload: Colab wipes uploaded files every time the
runtime disconnects, and this is a 105 MB re-upload each time. A file in Drive is
there permanently — upload it once and every future session just mounts it.

**To upload it:** open [drive.google.com](https://drive.google.com) in a new tab and
drag `grove_dataset.tgz` from your Desktop into the file list. Leave it in *My Drive*
(the top level) — that is where this cell looks.

Running the cell pops up a permission prompt. Approve it; that is Colab asking to
read your Drive.

In [ ]:
from google.colab import drive
import os, glob, shutil

drive.mount('/content/drive')

# look in My Drive root first, then anywhere in Drive
hits = glob.glob('/content/drive/MyDrive/grove_dataset.tgz') \
    or glob.glob('/content/drive/MyDrive/**/grove_dataset.tgz', recursive=True)
assert hits, ("grove_dataset.tgz not found in your Drive.\n"
              "Upload it to drive.google.com (top level of My Drive), then re-run.")
print("found:", hits[0], "(%.0f MB)" % (os.path.getsize(hits[0])/1048576))

os.chdir('/content')
if not os.path.exists('grove_dataset.tgz'):
    shutil.copy(hits[0], '/content/grove_dataset.tgz')
!tar xzf /content/grove_dataset.tgz -C /content
!sed -i "s|^path:.*|path: /content/grove_dataset|" /content/grove_dataset/data.yaml
!cat /content/grove_dataset/data.yaml

print("train:", len(glob.glob('/content/grove_dataset/train/images/*.jpg')),
      " val:", len(glob.glob('/content/grove_dataset/val/images/*.jpg')))

### Look at the labels before training

Never train on labels you have not seen. This draws a few so you can check the
boxes sit on real trees and shrubs.

In [ ]:
import cv2, random, matplotlib.pyplot as plt
names = open("grove_dataset/classes.txt").read().split()
COL = [(10,122,69), (62,160,217), (208,91,111)]      # trees, shrubs, people (BGR)
fig, axes = plt.subplots(1, 3, figsize=(17,4))
for ax, p in zip(axes, random.sample(glob.glob("grove_dataset/train/images/*.jpg"), 3)):
    im = cv2.imread(p); H, W = im.shape[:2]
    lp = p.replace("/images/","/labels/").replace(".jpg",".txt")
    for line in open(lp):
        ci, cx, cy, bw, bh = line.split()
        ci = int(ci); cx, cy, bw, bh = float(cx)*W, float(cy)*H, float(bw)*W, float(bh)*H
        x1, y1 = int(cx-bw/2), int(cy-bh/2)
        cv2.rectangle(im, (x1,y1), (int(cx+bw/2), int(cy+bh/2)), COL[ci], 2)
        cv2.putText(im, names[ci], (x1+2, max(12,y1-4)), cv2.FONT_HERSHEY_SIMPLEX, .45, COL[ci], 1)
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Train YOLOv8

`yolov8s` at 640 px — the same shape as Hailo's prebuilt models, and a good size
for the Hailo-8.

The labels came from RF-DETR, so this model **copies** it: the goal is the same
detections roughly 100× faster, not better ones. Expect mAP50 around 0.7–0.9
against its teacher; much lower means something is wrong with the labels.

In [ ]:
!pip -q install ultralytics
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
model.train(data="/content/grove_dataset/data.yaml",
            epochs=80, imgsz=640, batch=16, patience=20,
            project="grove", name="hat", exist_ok=True)
m = model.val()
print("mAP50:", round(m.box.map50,3), " mAP50-95:", round(m.box.map,3))

## 4. Export to ONNX

Opset 11, fixed batch — what the Hailo compiler expects.

In [ ]:
YOLO("grove/hat/weights/best.pt").export(
    format="onnx", imgsz=640, opset=11, simplify=True, dynamic=False)
!ls -la grove/hat/weights/*.onnx

## 5. Install the Hailo compiler

**The one manual step.** The Dataflow Compiler is not on PyPI. Register free at
[hailo.ai/developer-zone](https://hailo.ai/developer-zone/), then download:

* **Dataflow Compiler** — a `.whl`
* **Hailo Model Zoo** — a `.whl`

Take the **4.20.x** versions. The Pi runs HailoRT 4.20.0, and a newer compiler
produces a `.hef` it will refuse to load.

Upload both `.whl` files with the file browser, then run this.

In [ ]:
import glob
dfc = glob.glob("/content/hailo_dataflow_compiler-*.whl")
mz  = glob.glob("/content/hailo_model_zoo-*.whl")
assert dfc, "Upload the Dataflow Compiler .whl first"
!pip -q install {dfc[0]}
if mz: !pip -q install {mz[0]}
else:  !pip -q install git+https://github.com/hailo-ai/hailo_model_zoo.git
!hailo --version

## 6. Compile to `.hef`

Quantisation picks number ranges from a calibration set. Using **your own frames**
rather than stock images matters — the model sees your lighting and your campus.

In [ ]:
import pathlib, shutil, random
calib = pathlib.Path("calib"); calib.mkdir(exist_ok=True)
imgs = glob.glob("grove_dataset/train/images/*.jpg")
random.seed(0)
for p in random.sample(imgs, min(256, len(imgs))):
    shutil.copy(p, calib/pathlib.Path(p).name)
print("calibration images:", len(list(calib.glob('*'))))

In [ ]:
# --hw-arch hailo8 is required: hailo8l builds for the 13 TOPS part
!hailomz compile yolov8s \
    --ckpt grove/hat/weights/best.onnx \
    --hw-arch hailo8 \
    --calib-path calib \
    --classes 3 \
    --performance
!ls -la *.hef

## 7. Put it on the car

Download `yolov8s.hef` (file browser → right-click → Download), then from your Mac:

```bash
scp ~/Downloads/yolov8s.hef robocar:~/oakd_project/models/grove.hef
scp ~/Desktop/classes.txt   robocar:~/oakd_project/models/grove_classes.txt
ssh robocar 'python3 ~/oakd_project/hailo_backend.py'
```

That last command is a self-test — it runs the new model on real recorded frames
and prints what it found and how fast.

Nothing else changes. `live_survey.py` checks for `grove.hef` on every run and
uses the HAT automatically when it is there, falling back to the CPU when it is not.

**Expected: ~60 fps instead of 0.58** — detection keeps pace with the car instead
of trailing it, and CPU draw drops from 10–12 W to about 2.5 W.

### If the Pi rejects it
* `HEF format version mismatch` → compiler newer than HailoRT 4.20.0. Rebuild with 4.20.x.
* `Failed to parse HEF` → built for `hailo8l`. Rebuild with `--hw-arch hailo8`.
* Wrong class names → `grove_classes.txt` order must match `classes.txt` (trees, shrubs, people).


## What this does not fix

It makes detection fast. It does **not** change what the counts mean — they are
still clusters of the *rover's own position* when a detection fired, so they still
measure stretches of route rather than surveyed objects, and they still move when
you change the merge radius.

That needs depth from the OAK-D, not a faster detector.
